In [51]:
from __future__ import annotations

import operator
from typing import TypedDict, List, Annotated

from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv

load_dotenv()

True

In [52]:
class Task(BaseModel):
    id: int
    title: str
    brief: str = Field(...,description="What is to be covered")

In [53]:
class Plan(BaseModel):
    blog_title: str
    tasks: List[Task]

In [54]:
class State(TypedDict):
    topic: str
    plan: Plan

    sections: Annotated[List[str],operator.add]
    final: str

In [55]:
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

In [56]:
def orchestrator(state: State) -> dict:
    plan = llm.with_structured_output(Plan).invoke(
        [
            SystemMessage(
                content=(
                    "Create a blog plan with 5-7 sections on the following topic."
                )
            ),
            HumanMessage(content=f"Topic: {state['topic']}")
        ]
    )
    return {"plan":plan}

In [57]:
def fanout(state:State):
    return [Send('worker',{"task":task,"topic":state['topic'],"plan":state['plan']})
            for task in state['plan'].tasks]

In [58]:
def worker(payload: dict) -> dict:
    task = payload['task']
    topic = payload['topic']
    plan = payload['plan']

    blog_title = plan.blog_title

    section_md = llm.invoke(
        [
            SystemMessage(content='Write a single MD section for a blog'),
            HumanMessage(content=(
                f'Blog title: f{blog_title}\n'
                f'Topic: f{topic}\n'
                f'Section: f{task.title}\n'
                f'Brief: f{task.brief}\n\n'
                "Return content only for the section filed in this block in pure MD"
                )
            ),
        ]
    ).content.strip()
    return {"sections":[section_md]}

In [59]:
from pathlib import Path
import re

def reducer(state: State) -> dict:
    title = state['plan'].blog_title
    body = '\n\n'.join(state['sections']).strip()

    final_md = f'# {title}\n\n{body}\n'

    safe_title = re.sub(r'[^a-z0-9]+', '_', title.lower()).strip('_')
    filename = safe_title + '.md'

    output_path = Path(filename)
    output_path.write_text(final_md,encoding='utf-8')

    return{"final":final_md}

In [60]:
g = StateGraph(State)
g.add_node('orchestrator',orchestrator)
g.add_node('worker',worker)
g.add_node('reducer',reducer)

In [61]:
g.add_edge(START,'orchestrator')
g.add_conditional_edges('orchestrator',fanout,['worker'])
g.add_edge('worker','reducer')
g.add_edge('reducer',END)

app = g.compile()

In [ ]:
out = app.invoke({'topic':'write a blog on diffrent types of agents made using langchain','sections':[]})

In [ ]:
print(out['final'])

#Mastering LangChain Agents: A Deep Dive into Different Types and Their Applications

## Introduction: Unlocking Dynamic AI with LangChain Agents

In the rapidly evolving landscape of artificial intelligence, the ability to build systems that can not only understand but also act intelligently in complex, dynamic environments is paramount. This is precisely where **LangChain agents** emerge as a powerful and transformative paradigm. At their core, LangChain agents represent an advanced approach to creating autonomous AI systems, moving beyond simple question-answering to enable sophisticated, multi-step problem-solving.

So, what exactly is an agent in the LangChain context? Imagine an intelligent system where a Large Language Model (LLM) acts as the "brain." This LLM is not just generating text; it's reasoning, planning, and deciding. Its primary function is to determine which "tools" to use—be it a search engine, a calculator, a database query, or even custom APIs—to accomplish a give